# BioRefusalAudit on Colab T4 â€” bigger models via free GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SolshineCode/Deleeuw-AI-x-Bio-hackathon/blob/main/notebooks/colab_biorefusalaudit.ipynb)

Runs the full BioRefusalAudit eval pipeline on a Colab T4 (16 GB VRAM), covering models the local 4 GB GTX 1650 cannot fit:

- **Gemma 2 9B-IT** + **Gemma Scope 1 9B residual SAEs** (layer 20, width 16k)
- **Llama 3.1 8B-Instruct** + **Llama Scope residual SAEs**

Both models run at bnb 4-bit on T4. Estimated wall clock: ~60-90 minutes per model on a free T4.

## Key findings from local 2B-IT runs (confirmed before this notebook)

- **0 genuine refusals** across 75 prompts at both 80 and 200 token budgets (corrected after judge-consortium failure; see `TROUBLESHOOTING.md`)
- **Hazard-adjacent tier: 100% hedge** at both token budgets â€” behavioral posture is token-budget-stable
- **Named-circuit qualification: 60/75 (80%)** with inverted tier ordering (benign 87% > dual-use 80% > hazard-adjacent 73%)
- Mean divergence by tier (200 tok): benign 0.467, dual-use 0.655, hazard-adjacent 0.669

This notebook extends those findings to 9B-scale models as a cross-architecture check.

**Interactive demo** (explore results in browser, no server needed): `demo/interactive_explorer.html`

## Structure

1. Setup: check GPU, install deps, clone repo
2. Auth: HF\_TOKEN + GITHUB\_TOKEN from Colab secrets
3. Run: Gemma 2 9B-IT smoke test -> full eval -> regex rejudge
4. Run: Llama 3.1 8B-Instruct full eval -> regex rejudge
5. Cross-model scaling figure
6. Upload results to HuggingFace (or download as zip)


## 1. Setup + dependencies

In [ ]:
# Runtime check â€” make sure we got a GPU.
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# If GPU is not assigned, stop here. Runtime â†’ Change runtime type â†’ T4 GPU.
assert torch.cuda.is_available(), 'Need a T4 GPU. Runtime â†’ Change runtime type â†’ T4 GPU, then rerun.'

In [ ]:
# Install deps. sae_lens carries torch + transformers, so we don't pin those separately.
!pip install -q sae_lens bitsandbytes accelerate huggingface_hub click pyyaml matplotlib numpy

## 2. Authentication

Set these as Colab userdata secrets before running (left sidebar â†’ key icon):
- `HF_TOKEN` â€” HuggingFace token with Gemma + Llama license acceptance
- `GITHUB_TOKEN` â€” optional, only if pushing results back from the notebook

In [ ]:
from google.colab import userdata
from huggingface_hub import login as hf_login
import os

hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Set HF_TOKEN in Colab userdata secrets first.'
hf_login(token=hf_token)
os.environ['HF_TOKEN'] = hf_token
print('HF auth ok')

try:
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token:
        os.environ['GITHUB_TOKEN'] = gh_token
        print('GitHub token present â€” will push results at end')
    else:
        print('No GITHUB_TOKEN â€” will download results instead of pushing')
except Exception:
    print('No GITHUB_TOKEN â€” will download results instead of pushing')

## 3. Clone the repo

In [ ]:
import subprocess
import shutil
from pathlib import Path

REPO = 'SolshineCode/Deleeuw-AI-x-Bio-hackathon'
BRANCH = 'main'
WORK = Path('/content/Deleeuw-AI-x-Bio-hackathon')

# Force-clean any partial/stale clone from a prior Colab session.
# This prevents exit 128 ('destination path already exists and is not empty').
if WORK.exists():
    print(f'Removing existing {WORK} ...')
    shutil.rmtree(WORK)

import os
_gh_tok = os.environ.get('GITHUB_TOKEN', '')
_clone_url = (
    f'https://{_gh_tok}@github.com/{REPO}.git' if _gh_tok
    else f'https://github.com/{REPO}.git'
)
subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, _clone_url, str(WORK)],
    check=True
)

%cd /content/Deleeuw-AI-x-Bio-hackathon
!git log --oneline -3


In [ ]:
# Install the package itself so we can import biorefusalaudit
!pip install -q -e .

## 4. Eval: Gemma 2 9B-IT + Gemma Scope 1 residual SAEs

Gemma 2 9B-IT at bnb 4-bit â‰ˆ 5 GB, fits T4 comfortably. Gemma Scope 1 9B residual SAE at layer 20, width 16k.

Wall clock: ~45 minutes for 75 prompts on T4.

In [ ]:
# Smoke test on 3 prompts first.
!mkdir -p runs/colab_gemma-2-9b-it-smoke
!python -m biorefusalaudit.cli run \
    --model google/gemma-2-9b-it \
    --eval-set data/eval_set_public/eval_set_public_v1.jsonl \
    --out runs/colab_gemma-2-9b-it-smoke \
    --sae-source gemma_scope_1 \
    --sae-release gemma-scope-9b-pt-res \
    --sae-id "layer_20/width_16k/average_l0_138" \
    --layer 20 \
    --quantize 4bit \
    --no-llm-judges \
    --limit 3 2>&1 | tee runs/colab_gemma-2-9b-it-smoke/stderr.log | tail -40

In [ ]:
# Full run on Gemma 2 9B-IT. Uses the same 75-prompt eval set.
!mkdir -p runs/colab_gemma-2-9b-it
!python -m biorefusalaudit.cli run \
    --model google/gemma-2-9b-it \
    --eval-set data/eval_set_public/eval_set_public_v1.jsonl \
    --out runs/colab_gemma-2-9b-it \
    --sae-source gemma_scope_1 \
    --sae-release gemma-scope-9b-pt-res \
    --sae-id "layer_20/width_16k/average_l0_138" \
    --layer 20 \
    --quantize 4bit \
    --no-llm-judges 2>&1 | tee runs/colab_gemma-2-9b-it/stderr.log | tail -60


In [ ]:
# Regex re-judge G2 9B results (avoids uniform-prior judge-failure artifact).
# Mirrors the local rejudge pipeline; corrected report written to a -rejudged/ subdir.
!python scripts/rejudge_stored_completions.py \
    --report runs/colab_gemma-2-9b-it/report.json \
    --out    runs/colab_gemma-2-9b-it-rejudged/report.json

import json
from collections import Counter
data = json.load(open('runs/colab_gemma-2-9b-it-rejudged/report.json'))
recs = data.get('records', [])
labels = Counter(r['surface_label'] for r in recs)
print('G2 9B-IT corrected surface labels:', dict(labels))
print('Tier breakdown:')
for tier in ('benign_bio', 'dual_use_bio', 'hazard_adjacent_category'):
    tier_recs = [r for r in recs if r.get('tier') == tier]
    t_labels = Counter(r['surface_label'] for r in tier_recs)
    print(f'  {tier}: {dict(t_labels)}')


## 5. Eval: Llama 3.1 8B-Instruct + Llama Scope SAEs

Llama 3.1 8B-Instruct at bnb 4-bit â‰ˆ 4.5 GB. Llama Scope covers layers 0â€“31 with TopK SAEs at kâˆˆ{32, 64, 128}.

Cross-architecture sanity check for the scaling story.

In [ ]:
!mkdir -p runs/colab_llama-3.1-8b-it
!python -m biorefusalaudit.cli run \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --eval-set data/eval_set_public/eval_set_public_v1.jsonl \
    --out runs/colab_llama-3.1-8b-it \
    --sae-source llama_scope \
    --sae-release fnlp/Llama3_1-8B-Base-LXR-8x \
    --sae-id "l16r_8x" \
    --layer 16 \
    --quantize 4bit \
    --no-llm-judges 2>&1 | tee runs/colab_llama-3.1-8b-it/stderr.log | tail -60


In [ ]:
# Regex re-judge Llama results.
!python scripts/rejudge_stored_completions.py \
    --report runs/colab_llama-3.1-8b-it/report.json \
    --out    runs/colab_llama-3.1-8b-it-rejudged/report.json

import json
from collections import Counter
data = json.load(open('runs/colab_llama-3.1-8b-it-rejudged/report.json'))
recs = data.get('records', [])
labels = Counter(r['surface_label'] for r in recs)
print('Llama 3.1 8B corrected surface labels:', dict(labels))
print('Tier breakdown:')
for tier in ('benign_bio', 'dual_use_bio', 'hazard_adjacent_category'):
    tier_recs = [r for r in recs if r.get('tier') == tier]
    t_labels = Counter(r['surface_label'] for r in tier_recs)
    print(f'  {tier}: {dict(t_labels)}')


## 6. Cross-model scaling figure

Compose a quick matplotlib plot aggregating all available runs under `runs/`.

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

run_roots = sorted(p for p in Path('runs').glob('*/report.json') if '-smoke' not in p.parent.name)
print('Found reports:', [str(p) for p in run_roots])

models = []
tiers = ('benign_bio', 'dual_use_bio', 'hazard_adjacent_category')
data = {t: [] for t in tiers}
for rp in run_roots:
    payload = json.loads(rp.read_text())
    models.append(Path(rp).parent.name)
    for t in tiers:
        data[t].append(payload['aggregate'].get(t, {}).get('mean_divergence', 0.0))

import numpy as np
x = np.arange(len(models))
w = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
for i, t in enumerate(tiers):
    ax.bar(x + (i - 1) * w, data[t], w, label=t)
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=30, ha='right')
ax.set_ylabel('Mean divergence D')
ax.set_title('BioRefusalAudit â€” divergence by tier Ã— model')
ax.legend()
plt.tight_layout()

out_dir = Path('runs/cross_model')
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 'scaling_plot.png', dpi=150)
print('Saved', out_dir / 'scaling_plot.png')
plt.show()

## 7. Save results to HuggingFace

Requires `HF_TOKEN` (write scope) in Colab Secrets (left sidebar, key icon).
Creates a **private** dataset repo `Solshine/biorefusalaudit-colab-runs` and uploads
all `report.json` files plus the scaling figure. Re-runs append safely (`exist_ok=True`).

The fallback cell below downloads everything as a local zip if HF_TOKEN is unavailable.

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path
import os

HF_RESULTS_REPO = "Solshine/biorefusalaudit-colab-runs"

hf_tok = os.environ.get("HF_TOKEN")
if not hf_tok:
    print("No HF_TOKEN -- skipping HF upload. Use the zip-download cell below.")
else:
    api = HfApi()
    # Private dataset repo; exist_ok so re-runs don't error
    api.create_repo(HF_RESULTS_REPO, repo_type="dataset", private=True, exist_ok=True)
    print(f"HF repo: https://huggingface.co/datasets/{HF_RESULTS_REPO}")

    uploaded = []

    for rp in sorted(Path("runs").rglob("report.json")):
        dest = f"{rp.parent.name}/report.json"
        api.upload_file(path_or_fileobj=str(rp), path_in_repo=dest,
                        repo_id=HF_RESULTS_REPO, repo_type="dataset")
        uploaded.append(dest)
        print(f"  uploaded {dest}")

    for lp in sorted(Path("runs").rglob("stderr.log")):
        dest = f"{lp.parent.name}/stderr.log"
        api.upload_file(path_or_fileobj=str(lp), path_in_repo=dest,
                        repo_id=HF_RESULTS_REPO, repo_type="dataset")
        uploaded.append(dest)

    scaling = Path("runs/cross_model/scaling_plot.png")
    if scaling.exists():
        api.upload_file(path_or_fileobj=str(scaling),
                        path_in_repo="cross_model/scaling_plot.png",
                        repo_id=HF_RESULTS_REPO, repo_type="dataset")
        uploaded.append("cross_model/scaling_plot.png")
        print("  uploaded cross_model/scaling_plot.png")

    print(f"Done. {len(uploaded)} artifact(s) saved to private HF repo:")
    print(f"  https://huggingface.co/datasets/{HF_RESULTS_REPO}")


In [ ]:
# Alternative: zip + download the runs directory to your local machine.
import shutil
shutil.make_archive('/content/colab_runs', 'zip', 'runs')
from google.colab import files
files.download('/content/colab_runs.zip')